# KarcinoJen Colab Runner

This notebook executes the paper architecture in the intended modality order:
1. Multimodal retrieval (ColPali MaxSim + lexical fusion).
2. VLM extraction from retrieved page images (Gemini primary).
3. Deterministic SVD validation + CoVe retries.
4. Text-LLM synthesis/enrichment (Groq primary, local fallback).

A lexical retrieval run is included as an ablation/fallback profile only.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path("/content/KarcinoJen")
REPO_URL = os.environ.get("KARCINOJEN_REPO_URL", "").strip()

if PROJECT_ROOT.exists() and (PROJECT_ROOT / "scripts" / "run_pipeline.py").exists():
    print(f"Using existing repo at {PROJECT_ROOT}")
elif REPO_URL:
    print(f"Cloning from {REPO_URL}")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / "scripts" / "run_pipeline.py").exists():
        raise RuntimeError("Repository not found. Set env var KARCINOJEN_REPO_URL, then rerun this cell.")

os.chdir(PROJECT_ROOT)
print(f"Working directory: {PROJECT_ROOT}")

In [ ]:
#!/usr/bin/env python3
"""Ensure data and models are available on Colab."""
from pathlib import Path
import subprocess

PROJECT_ROOT = Path("/content/KarcinoJen")

# Check if ColPali index exists; if not on Colab, build it
colpali_index_dir = PROJECT_ROOT / "data" / "colpali_index"
datasheets_dir = PROJECT_ROOT / "data" / "datasheets"
svd_dir = PROJECT_ROOT / "data" / "svd"

print("Verifying data structure...")
for data_dir, name in [(datasheets_dir, "Datasheets"),
                        (svd_dir, "SVD files"),
                        (colpali_index_dir, "ColPali index")]:
    if data_dir.exists():
        count = len(list(data_dir.glob("*")))
        print(f"  ✓ {name}: {count} files found")
    else:
        print(f"  ⚠ {name}: not found (will be generated on first run)")

# On Colab, ColPali embeddings will be auto-generated on first retrieval
# This may take 2-5 minutes depending on GPU
print("\nNote: ColPali visual embeddings will be generated on first retrieval.")
print("This is a one-time process; subsequent runs will use cached embeddings.")

In [ ]:
!python -m pip install --upgrade pip

# Install PyTorch with CUDA 11.8 support (compatible with Colab GPUs)
# This ensures ColPali can use GPU acceleration for embedding generation
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Install project requirements (uses torch with CUDA already installed)
!python -m pip install -r requirements.txt

print("✓ Dependencies installed with CUDA support")

In [ ]:
#!/usr/bin/env python3
"""Check and configure CUDA for Google Colab GPU runtime."""
import subprocess
import torch

print("=" * 70)
print("CUDA and GPU Setup for KarcinoJen on Google Colab")
print("=" * 70)

# Check NVIDIA GPU availability
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print("\n✓ NVIDIA GPU detected:")
    print(result.stdout[:500])  # First 500 chars of output
else:
    print("\n⚠ WARNING: No NVIDIA GPU detected. Ensure runtime type is set to GPU:")
    print("   Runtime → Change runtime type → GPU (A100, V100, or T4)")

# Verify PyTorch CUDA availability
cuda_available = torch.cuda.is_available()
cuda_version = torch.version.cuda if cuda_available else "N/A"
device_count = torch.cuda.device_count() if cuda_available else 0

print(f"\nPyTorch CUDA Info:")
print(f"  CUDA available: {cuda_available}")
print(f"  CUDA version:   {cuda_version}")
print(f"  GPU count:      {device_count}")

if cuda_available:
    print(f"  GPU name:       {torch.cuda.get_device_name(0)}")
    print(f"  GPU memory:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("\n✓ GPU is ready for ColPali indexing and VLM inference!")
else:
    print("\n⚠ CUDA not available. Switching to CPU mode (slower).")
    print("   CPU-mode ColPali will still work but be slower.")

print("\n" + "=" * 70)

In [ ]:
import getpass
import os

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter GEMINI_API_KEY: ")

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter GROQ_API_KEY (for synthesis): ")

print("API keys loaded into environment for this runtime session.")

In [ ]:
import json
from pathlib import Path

CONFIG_PATH = Path("configs/model_config.json")

def set_runtime_profile(*, retrieval_backend: str = "colpali", selected_provider: str = "gemini") -> None:
    cfg = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    cfg["selected_provider"] = selected_provider
    cfg["retrieval"]["backend"] = retrieval_backend

    # Keep canonical stage split explicit.
    cfg["providers"]["gemini"]["api_key_env"] = "GEMINI_API_KEY"
    cfg["providers"]["groq"]["api_key_env"] = "GROQ_API_KEY"

    CONFIG_PATH.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
    print(f"Updated config: selected_provider={selected_provider}, retrieval.backend={retrieval_backend}")

# Canonical paper profile
set_runtime_profile(retrieval_backend="colpali", selected_provider="gemini")

In [ ]:
DATASHEET = "data/datasheets/stm32f401-rm.pdf"
QUERY = (
    "Extract USART2 CR1 control bits including UE, M, PCE, TE, and RE with bit positions."
)
TOP_K = "5"

print("Datasheet:", DATASHEET)
print("Query:", QUERY)

In [ ]:
cmd = [
    sys.executable,
    "scripts/run_pipeline.py",
    "--datasheet",
    DATASHEET,
    "--query",
    QUERY,
    "--top-k",
    TOP_K,
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

## Optional Ablation: Resource-Constrained Retrieval

Use this only for fallback experiments and local low-VRAM runs.
The VLM stage still consumes rendered page images, but retrieval is lexical instead of ColPali.

In [ ]:
set_runtime_profile(retrieval_backend="lexical", selected_provider="gemini")
ablation_cmd = [
    sys.executable,
    "scripts/run_tests.py",
    "--backend",
    "lexical",
    "--case",
    "2",
]
print("Running ablation:", " ".join(ablation_cmd))
subprocess.run(ablation_cmd, check=False)

## Artifacts

Expected generated outputs:
- generated/drivers/<timestamp>/driver.h
- generated/drivers/<timestamp>/driver.c
- generated/drivers/<timestamp>/audit_trace.json

Use the canonical ColPali profile for main paper claims and the lexical profile as an ablation reference.